# ENGR 691-F Assignment 1: Dynamic Programming on FrozenLake
## Complete Submission Package

**Instructions:**
1. Run all cells (Ctrl+A, then Shift+Enter)
2. Copy the results printed in the console
3. Fill in the report template at the bottom
4. Save as PDF and submit to Moodle

---

In [ ]:
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("Imports successful!")

## Core Functions

In [ ]:
def create_env():
    env = gym.make("FrozenLake-v1", is_slippery=True)
    return env

def get_model(env):
    P = env.unwrapped.P
    nS = env.observation_space.n
    nA = env.action_space.n
    return P, nS, nA

def one_step_lookahead(P, s, V, nA, gamma):
    q = np.zeros(nA)
    for a in range(nA):
        for prob, s_next, reward, done in P[s][a]:
            if done:
                q[a] += prob * reward
            else:
                q[a] += prob * (reward + gamma * V[s_next])
    return q

def greedy_policy_from_value(P, V, nS, nA, gamma):
    policy = np.zeros(nS, dtype=int)
    for s in range(nS):
        q = one_step_lookahead(P, s, V, nA, gamma)
        policy[s] = np.argmax(q)
    return policy

def evaluate_policy(env, policy, episodes=1000, max_steps=200):
    successes = 0
    returns = []
    success_steps = []

    for _ in range(episodes):
        state, _ = env.reset()
        total_reward = 0
        steps = 0
        done = False

        while not done and steps < max_steps:
            action = policy[state]
            next_state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            steps += 1
            done = terminated or truncated
            state = next_state

        returns.append(total_reward)
        if total_reward > 0:
            successes += 1
            success_steps.append(steps)

    success_rate = successes / episodes
    avg_return = np.mean(returns)
    avg_steps_success = np.mean(success_steps) if success_steps else np.nan
    return success_rate, avg_return, avg_steps_success

print("Core functions defined!")

## Value Iteration Implementation

In [ ]:
def value_iteration(env, gamma=0.99, tol=1e-8, max_iter=1000, eval_episodes=500):
    P, nS, nA = get_model(env)
    V = np.zeros(nS)

    delta_history = []
    success_history = []
    return_history = []

    for k in range(max_iter):
        V_new = np.zeros(nS)

        for s in range(nS):
            q = one_step_lookahead(P, s, V, nA, gamma)
            V_new[s] = np.max(q)

        delta = np.max(np.abs(V_new - V))
        delta_history.append(delta)

        policy_k = greedy_policy_from_value(P, V_new, nS, nA, gamma)
        sr, ar, _ = evaluate_policy(env, policy_k, episodes=eval_episodes)
        success_history.append(sr)
        return_history.append(ar)

        if (k+1) % 5 == 0:
            print(f"VI Iteration {k+1}: δ = {delta:.2e}, SR = {sr:.4f}, AR = {ar:.4f}")

        V = V_new

        if delta < tol:
            print(f"\n✓ VI Converged at iteration {k+1}\n")
            break

    final_policy = greedy_policy_from_value(P, V, nS, nA, gamma)
    final_sr, final_ar, final_steps = evaluate_policy(env, final_policy, episodes=2000)

    return {
        "V": V,
        "policy": final_policy,
        "delta_history": delta_history,
        "success_history": success_history,
        "return_history": return_history,
        "final_success_rate": final_sr,
        "final_avg_return": final_ar,
        "final_avg_steps_success": final_steps,
        "iterations": len(delta_history)
    }

print("Value Iteration function defined!")

## Asynchronous Value Iteration Implementation

In [ ]:
def async_value_iteration(env, gamma=0.99, tol=1e-8, max_iter=1000, eval_episodes=500):
    P, nS, nA = get_model(env)
    V = np.zeros(nS)
    delta_history = []
    success_history = []
    return_history = []

    for k in range(max_iter):
        delta = 0.0
        for s in range(nS):
            old_v = V[s]
            q = one_step_lookahead(P, s, V, nA, gamma)
            V[s] = np.max(q)
            delta = max(delta, abs(V[s] - old_v))

        delta_history.append(delta)
        
        policy_k = greedy_policy_from_value(P, V, nS, nA, gamma)
        sr, ar, _ = evaluate_policy(env, policy_k, episodes=eval_episodes)
        success_history.append(sr)
        return_history.append(ar)

        if (k+1) % 5 == 0:
            print(f"AVI Iteration {k+1}: δ = {delta:.2e}, SR = {sr:.4f}, AR = {ar:.4f}")

        if delta < tol:
            print(f"\n✓ AVI Converged at iteration {k+1}\n")
            break

    final_policy = greedy_policy_from_value(P, V, nS, nA, gamma)
    final_sr, final_ar, final_steps = evaluate_policy(env, final_policy, episodes=2000)

    return {
        "V": V,
        "policy": final_policy,
        "delta_history": delta_history,
        "success_history": success_history,
        "return_history": return_history,
        "final_success_rate": final_sr,
        "final_avg_return": final_ar,
        "final_avg_steps_success": final_steps,
        "iterations": len(delta_history)
    }

print("Async Value Iteration function defined!")

## RUN THE ALGORITHMS

In [ ]:
print("="*70)
print("ENGR 691-F Assignment 1: Dynamic Programming on FrozenLake")
print("="*70)

env = create_env()
P, nS, nA = get_model(env)
print(f"\nEnvironment: FrozenLake-v1 (Slippery)")
print(f"States: {nS} | Actions: {nA}")

gamma = 0.99
tol = 1e-8
max_iter = 1000

print("\n" + "="*70)
print("VALUE ITERATION")
print("="*70)
vi_results = value_iteration(env, gamma=gamma, tol=tol, max_iter=max_iter, eval_episodes=500)

print("\n" + "="*70)
print("ASYNCHRONOUS VALUE ITERATION")
print("="*70)
avi_results = async_value_iteration(env, gamma=gamma, tol=tol, max_iter=max_iter, eval_episodes=500)

## Display Results

In [ ]:
print("\n" + "="*70)
print("RESULTS - VALUE ITERATION")
print("="*70)
print(f"Converged in: {vi_results['iterations']} iterations")
print(f"Final Success Rate: {vi_results['final_success_rate']:.4f}")
print(f"Final Average Return: {vi_results['final_avg_return']:.4f}")
print(f"Avg Steps on Success: {vi_results['final_avg_steps_success']:.2f}")

print("\n" + "="*70)
print("RESULTS - ASYNCHRONOUS VALUE ITERATION")
print("="*70)
print(f"Converged in: {avi_results['iterations']} iterations")
print(f"Final Success Rate: {avi_results['final_success_rate']:.4f}")
print(f"Final Average Return: {avi_results['final_avg_return']:.4f}")
print(f"Avg Steps on Success: {avi_results['final_avg_steps_success']:.2f}")

print("\n" + "="*70)
print("COPY THESE VALUES INTO YOUR REPORT")
print("="*70)
print(f"\nVI iterations: {vi_results['iterations']}")
print(f"VI success rate: {vi_results['final_success_rate']:.3f}")
print(f"VI avg return: {vi_results['final_avg_return']:.3f}")
print(f"VI avg steps: {vi_results['final_avg_steps_success']:.2f}")
print(f"\nAVI iterations: {avi_results['iterations']}")
print(f"AVI success rate: {avi_results['final_success_rate']:.3f}")
print(f"AVI avg return: {avi_results['final_avg_return']:.3f}")
print(f"AVI avg steps: {avi_results['final_avg_steps_success']:.2f}")

## Generate Plots

In [ ]:
def plot_metric(values, ylabel, title, filename):
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(values) + 1), values, marker='o', markersize=4, linewidth=2, color='#2E86AB')
    plt.xlabel("Iteration", fontsize=12, fontweight='bold')
    plt.ylabel(ylabel, fontsize=12, fontweight='bold')
    plt.title(title, fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {filename}")

print("\nGenerating plots...\n")

plot_metric(
    vi_results["success_history"],
    ylabel="Success Rate",
    title="Success Rate vs Iteration (Value Iteration)",
    filename="success_rate_vs_iteration.png"
)

plot_metric(
    vi_results["return_history"],
    ylabel="Average Return",
    title="Average Return vs Iteration (Value Iteration)",
    filename="average_return_vs_iteration.png"
)

---

# REPORT TEMPLATE

**Copy everything below into a Word document and fill in the [INSERT: X] values from the results above.**

---

# ENGR 691-F: Assignment 1 Report
## Dynamic Programming on FrozenLake-v1

### Problem Setup

In this assignment, I solved the slippery FrozenLake-v1 environment using dynamic programming with access to the full transition model P = env.unwrapped.P. The goal was to compute an optimal value function using value iteration, extract the corresponding greedy policy, and evaluate that policy empirically through simulation. The implementation was written in a model-based way, without assuming a specific grid size, layout, or fixed transition probabilities. The environment contains 16 states (4×4 grid) and 4 actions (up, down, left, right), with stochastic transitions due to the slippery ice.

### Value Iteration Method

Value iteration updates each state value using the Bellman optimality equation by taking the maximum expected return over all actions. At every iteration k, I computed a new value function V_{k+1} from the previous estimate V_k. The algorithm stops when the convergence measure δ_k = max_s |V_{k+1}(s) - V_k(s)| < tolerance or when the maximum number of iterations is reached. Terminal transitions were handled carefully so that no future value was added after the episode had ended.

### Role of Delta (δ) as Stopping Criterion

The quantity δ_k is the largest change in the value function over all states at iteration k. It acts as a stopping criterion because it measures how much the value estimates are still changing. When δ_k becomes very small (below the tolerance threshold), the value function has nearly stabilized, so further updates produce little improvement. This provides a robust and theoretically justified way to detect convergence in dynamic programming.

### Greedy Policy Extraction and Evaluation

After convergence, I extracted the greedy policy by choosing, in each state, the action with the largest one-step lookahead value. I then evaluated this policy over multiple simulation episodes and reported:
- **Success Rate**: fraction of episodes that reached the goal
- **Average Return**: mean cumulative reward across all episodes
- **Average Steps on Success**: mean episode length conditioned on successful episodes

During value iteration, I evaluated the greedy policy at each iteration and plotted how policy performance improved over time.

### Comparison: Value Iteration vs Asynchronous Value Iteration

**Value Iteration (Synchronous):**
- Updates all states from the value function of the previous iteration
- Requires storing both V_k and V_{k+1}
- One sweep = one full update across all states

**Asynchronous Value Iteration (In-Place Updates):**
- Updates values in place; later states within the same sweep use newly updated values from earlier states
- Can propagate useful information faster because the newest value estimates are used immediately
- Typically requires fewer iterations to reach the same convergence threshold

In this experiment:
- **Value Iteration converged in: [INSERT: X] iterations**
- **Asynchronous Value Iteration converged in: [INSERT: X] iterations**

The asynchronous method often converges faster because in-place updates reuse fresh estimates, reducing the effective "lag" between old and new information. However, the final policy quality is typically similar between both methods.

### Why Access to the Model P Makes Dynamic Programming Possible

Dynamic programming is possible here because the environment provides the full model P[s][a], which contains, for every state-action pair (s,a):
- Transition probabilities P(s'|s,a) to each next state s'
- Immediate rewards r(s,a,s')
- Termination information (done flag)

With this complete model, the expected return of each action can be computed **exactly** using Bellman updates, rather than being estimated only from sampled experience. Without access to P, the algorithm would need model-free reinforcement learning methods such as Q-learning or SARSA, which estimate these expectations from random samples of the environment. With the model, dynamic programming computes the expectation exactly, leading to faster convergence and guaranteed optimality.

### Results Summary

**Value Iteration Performance:**
- Converged in: **[INSERT: X]** iterations
- Final Success Rate: **[INSERT: X.XXX]**
- Final Average Return: **[INSERT: X.XXX]**
- Average Steps to Goal (on success): **[INSERT: X.XX]**

**Asynchronous Value Iteration Performance:**
- Converged in: **[INSERT: X]** iterations
- Final Success Rate: **[INSERT: X.XXX]**
- Final Average Return: **[INSERT: X.XXX]**
- Average Steps to Goal (on success): **[INSERT: X.XX]**

Both methods discovered effective policies that successfully navigate the slippery FrozenLake environment. The success-rate and average-return plots demonstrate clear improvement in policy quality as iterations progress and the value function converges.

### Conclusion

Overall, this assignment demonstrates how model-based dynamic programming can compute an effective policy for FrozenLake by repeatedly applying Bellman optimality updates until the value function stabilizes, and how policy quality can be verified empirically through simulation. The stopping criterion δ provides a principled way to detect convergence, and the comparison between synchronous and asynchronous updates illustrates different computational trade-offs in DP. Access to the full model P enables exact computation of expected returns, making DP significantly more efficient than model-free methods on this problem.